In [16]:
import numpy as np
from sklearn.decomposition import PCA
import perturbot
#from perturbot.match.ott_egwl import get_coupling_egw_labels_ott
#from perturbot.predict import train_mlp
from perturbot.match import (
    get_coupling_cotl, 
    get_coupling_cotl_sinkhorn, 
    get_coupling_egw_labels_ott,
    get_coupling_egw_all_ott,
    get_coupling_eot_ott,
    get_coupling_leot_ott,
    get_coupling_egw_ott,
    get_coupling_cot, 
    get_coupling_cot_sinkhorn, 
    get_coupling_gw_labels,
    get_coupling_fot,
)
from perturbot.predict import train_mlp
import torch

In [47]:
X_dict[0].dtype

dtype('float64')

In [2]:
# Generate data
n_samples = 300
labels = [0,1,2,3]
X_dict = {k: np.random.rand(n_samples,1000) for k in labels}
Y_dict = {k: np.random.rand(n_samples,2000) for k in labels}
pca = PCA(n_components=50)
X_reduced = {k: pca.fit_transform(X_dict[k]) for k in labels}
Y_reduced = {k: pca.fit_transform(Y_dict[k]) for k in labels}


# Learn matching in the latent space
T_dict, log = get_coupling_egw_labels_ott((X_reduced, Y_reduced)) # Other get_coupling_X methods be used

# Train MLP based on matching
model, pred_log = train_mlp((X_dict, Y_dict), T_dict)

# Learn feature-feature matching
T_feature, fm_log = get_coupling_fot((X_dict, Y_dict), T_dict)

running EGWL with ott
GW called
lse step
updating linearization
Label considered for Sinkhorn run
lse step
updating linearization
Label considered for Sinkhorn run


/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speci ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


5 outer iterations were needed.
The last Sinkhorn iteration has converged: True
The outer loop of Gromov Wasserstein has converged: True
The final regularized GW cost is: 1.646
Done running LEGWOT with ott


/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:67: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speci ...

  | Name         | Type       | Params
--------

INFO     Running sanity check on val set...                                                                        


/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/speciesOT_env/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:293: The number of training batches (5) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs fo

Epoch 212/2000:  11%|█         | 212/2000 [01:36<13:33,  2.20it/s, v_num=3, train_loss_epoch=0.0841]
Monitored metric val_loss did not improve in the last 45 records. Best score: 0.096. Signaling Trainer to stop.
lse step
Delta: 0.00025085318905942154  Loss: 0.16663181270186347
lse step
Delta: 0.0  Loss: 0.16663181270186347
converged at iter  1


In [41]:
T_feature.shape

(1000, 2000)

In [28]:
X= X_dict[0]

In [30]:
model = model.float()
model.eval()
device = next(model.parameters()).device
X = torch.from_numpy(X.astype('float32')).to(next(model.parameters()).device)

with torch.inference_mode():
    logits = model(X)                # e.g. [N, num_classes]
    probs = torch.softmax(logits, -1)  # or torch.sigmoid for binary
    
preds = probs.cpu().numpy()          # [N, num_classes] (or [N, 1] for binary)
labels = preds.argmax(axis=-1)       # integer class labels (multi-class)